In [3]:
import pandas as pd
import os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Any
import numpy as np

class HealthcareRAG:
    def __init__(self, model_name="microsoft/Phi-3.5-mini-instruct", cache_dir=None, device="cuda"):
        """
        Initialize the Healthcare RAG system with Phi-3.5 model

        Args:
            model_name: The Phi-3.5 model to use
            cache_dir: Directory to cache models
            device: Device to run the model on ('cuda' or 'cpu')
        """
        self.device = "cuda" if torch.cuda.is_available() and device == "cuda" else "cpu"
        print(f"Using device: {self.device}")

        # Load tokenizer and model
        print("Loading Phi-3.5 model and tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            cache_dir=cache_dir
        ).to(self.device)

        # Load embedding model
        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2').to(self.device)

        # Initialize empty index and document store
        self.index = None
        self.documents = []

    def load_healthcare_magic_dataset(self, limit=10000):
        """
        Load and prepare Healthcare Magic dataset

        Args:
            limit: Number of rows to use (default: 10000)
        """
        print(f"Loading Healthcare Magic dataset (limit: {limit})...")
        dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")

        # Get the 'train' split and convert to DataFrame
        df = dataset['train'].to_pandas()

        # Select just what we need and limit rows
        df = df.head(limit)

        # Process the dataset
        self.documents = []
        for idx, row in df.iterrows():
            # Combine question and answer into a document
            document = {
                'id': idx,
                'question': row['input'],
                'answer': row['output'],
                'content': f"Question: {row['input']}\nAnswer: {row['output']}"
            }
            self.documents.append(document)

        print(f"Processed {len(self.documents)} documents from Healthcare Magic dataset")

    def build_index(self):
        """
        Build FAISS index from documents
        """
        print("Building search index...")

        # Create text chunks to embed
        texts = [doc['content'] for doc in self.documents]

        # Generate embeddings
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True)

        # Normalize embeddings
        embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

        # Build FAISS index
        vector_dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(vector_dimension)
        self.index.add(embeddings.astype('float32'))

        print(f"Index built with {self.index.ntotal} vectors of dimension {vector_dimension}")

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: User query
            top_k: Number of documents to retrieve

        Returns:
            List of relevant documents
        """
        # Generate embedding for the query
        query_embedding = self.embedding_model.encode([query], show_progress_bar=False)
        query_embedding = query_embedding / np.linalg.norm(query_embedding)

        # Search the index
        scores, indices = self.index.search(query_embedding.astype('float32'), top_k)

        # Retrieve matching documents
        results = []
        for i, idx in enumerate(indices[0]):
            if idx != -1 and idx < len(self.documents):  # Valid index
                doc = self.documents[idx]
                results.append({
                    'id': doc['id'],
                    'content': doc['content'],
                    'question': doc['question'],
                    'answer': doc['answer'],
                    'score': float(scores[0][i])
                })

        return results

    def generate_response(self, query: str, retrieved_docs: List[Dict[str, Any]]) -> str:
      """
      Generate response using Phi-3.5 with retrieved context
      """
      # Sort retrieved docs by relevance score and format content
      sorted_docs = sorted(retrieved_docs, key=lambda x: x['score'], reverse=True)

      # Format context with relevance scores
      context_parts = []
      for i, doc in enumerate(sorted_docs):
          context_parts.append(f"[Document {i+1}] Relevance: {doc['score']:.2f}\nQ: {doc['question']}\nA: {doc['answer']}")

      context = "\n\n".join(context_parts)

      # Create improved prompt
      prompt = f"""You are a helpful healthcare assistant providing information based on medical knowledge.
  Answer the user's health question using ONLY the information from the retrieved documents below.

  Retrieved Documents:
  {context}

  User Question: {query}

  Guidelines:
  1. If the retrieved information doesn't contain a direct answer, say "I don't have enough information to answer this question confidently."
  2. Be concise and focused on addressing the main concern.
  3. Present symptoms, possible causes, and general treatments if available in the retrieved information.
  4. Always include a medical disclaimer reminding the user to consult a healthcare professional.

  Response:"""

  # Rest of the function remains the same...

      # Tokenize input
      inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

      # Generate response
      with torch.no_grad():
          outputs = self.model.generate(
              **inputs,
              max_new_tokens=512,
              temperature=0.7,
              top_p=0.9,
              do_sample=True
          )

      # Decode response
      response = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

      return response.strip()

    def query(self, user_query: str, top_k: int = 5) -> str:
        """
        End-to-end RAG pipeline: retrieve + generate

        Args:
            user_query: User question
            top_k: Number of documents to retrieve

        Returns:
            Generated response
        """
        # Retrieve relevant documents
        retrieved_docs = self.retrieve(user_query, top_k=top_k)

        # Generate response
        response = self.generate_response(user_query, retrieved_docs)

        return response


# Example usage
def main():
    # Initialize RAG system
    rag = HealthcareRAG(device="cuda")

    # Load dataset and build index
    rag.load_healthcare_magic_dataset(limit=10000)
    rag.build_index()

    # Interactive chat loop
    print("\nHealthcare Chatbot Ready! (Type 'exit' to quit)")
    print("=" * 50)

    while True:
        user_input = input("\nYour health question: ")
        if user_input.lower() in ['exit', 'quit', 'q']:
            break

        response = rag.query(user_input)
        print("\nChatbot:", response)
        print("\n" + "-" * 50)


if __name__ == "__main__":
    main()

Using device: cpu
Loading Phi-3.5 model and tokenizer...


Fetching 2 files:   0%|          | 0/2 [2:44:28<?, ?it/s]


OSError: microsoft/Phi-3.5-mini-instruct does not appear to have files named ('model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors'). Checkout 'https://huggingface.co/microsoft/Phi-3.5-mini-instruct/tree/main'for available files.